# Surrogate Model — Sample Workflow

This notebook walks through the **complete pipeline** for the `surrogate_model_clean` project:

| Stage | What happens |
|-------|--------------|
| 1 | **Data generation** — random graph weights → unnormalised quantum amplitude vectors |
| 2 | **Shard merging** — combine shard files into a single dataset |
| 3 | **Model training** — train a surrogate PNN (Polynomial Neural Network) |
| 4 | **Inverse design** — optimise graph weights toward a GHZ target state |
| 5 | **Output inspection** — list and summarise all generated files |

---

### This is a toy / demo run

All settings are deliberately tiny so the notebook completes in **under two minutes on a laptop CPU**.
The model accuracy and optimisation fidelity will be low — that is expected and intentional.

To run the real pipeline, increase the settings in the **Configuration** cell and then run the
command-line scripts (`src/data_generation.py`, `src/model_training.py`, `src/optimiser.py`)
on a machine with more memory and optionally a GPU.

### Prerequisites

Before running this notebook, make sure you have installed the project dependencies:

```bash
pip install -r requirements.txt
pip install jupyter ipykernel
```

Then launch from the **project root**:

```bash
jupyter notebook notebooks/sample_workflow.ipynb
```

---
## Setup

The cell below:
- auto-detects the project root (works from `notebooks/` or the project root)
- adds `src/` and `configs/` to `sys.path` so the project modules are importable
- imports all required modules
- creates the demo output tree under `results/sample_workflow/`

**No edits needed here.**

In [ ]:
import sys
import json
import importlib
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Auto-detect project root: works whether launched from the project root
# or from inside notebooks/
# ---------------------------------------------------------------------------
_cwd = Path.cwd()
if (_cwd / "src").exists() and (_cwd / "configs").exists():
    REPO_ROOT = _cwd
elif (_cwd.parent / "src").exists() and (_cwd.parent / "configs").exists():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Cannot locate the project root from {_cwd}.\n"
        "Run this notebook from the project root or from the notebooks/ folder."
    )

# Add src/ and configs/ to sys.path so project modules are importable.
for _p in [REPO_ROOT / "src", REPO_ROOT / "configs"]:
    _s = str(_p)
    if _s not in sys.path:
        sys.path.insert(0, _s)

# ---------------------------------------------------------------------------
# All demo outputs live under results/sample_workflow/
# ---------------------------------------------------------------------------
RESULTS_ROOT = REPO_ROOT / "results" / "sample_workflow"
DATA_DIR     = RESULTS_ROOT / "data"
MODEL_DIR    = RESULTS_ROOT / "model"
OPT_DIR      = RESULTS_ROOT / "optimisation"

for _d in [DATA_DIR, MODEL_DIR, OPT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Project imports
# ---------------------------------------------------------------------------
import numpy as np
import jax

print("Project root :", REPO_ROOT)
print("Results root :", RESULTS_ROOT)
print("Data dir     :", DATA_DIR)
print("Model dir    :", MODEL_DIR)
print("Optimiser dir:", OPT_DIR)
print()
print("JAX devices  :", jax.devices())
print("JAX backend  :", jax.default_backend())

---
## Configuration

All parameters for the demo run are defined in this single cell.
You can safely change the values marked **[editable]** to experiment.

| Parameter | Demo value | Note |
|-----------|-----------|------|
| `NODES` | 4 | Number of graph nodes / photons |
| `TARGET` | GHZ | Target quantum state |
| `N_SAMPLES` | 500 | Training samples to generate |
| `NUM_EPOCHS` | 15 | Training epochs |
| `OPT_STEPS` | 15 | Optimisation steps per sample |
| `OPT_SAMPLES` | 3 | Starting samples for the optimiser |

In [ ]:
# =============================================================================
# Main knobs — [editable]
# =============================================================================

NODES  = 4          # graph nodes / photons
TARGET = "GHZ"     # "GHZ" | "W" | "LINEAR_CLUSTER"

# =============================================================================
# Data generation — [editable]
# =============================================================================

N_SAMPLES      = 500    # total samples to generate
DATA_BATCH     = 100    # JAX batch size during generation
DATA_SEED      = 42     # reproducibility seed
SHARD_SIZE     = 500    # samples per .npz shard
NORMED_DATA    = False  # False = raw unnormalised amplitude vectors (project default)

# =============================================================================
# Model training — [editable]
# =============================================================================

MODEL_TYPE     = "PNN"  # "PNN" (Polynomial Neural Network) or "FNN"
HIDDEN_DIM     = 400    # PNN: integer; FNN: tuple e.g. (400, 400)
TRAIN_BATCH    = 50     # mini-batch size
NUM_EPOCHS     = 15     # training epochs  (use 10 000+ for production)
PATIENCE       = 8      # early-stop patience
LEARNING_RATE  = 1e-3

# =============================================================================
# Inverse design — [editable]
# =============================================================================

OPT_SAMPLES    = 3      # starting graph samples to optimise
OPT_STEPS      = 15     # gradient-descent steps per sample (use 10 000+ for production)

# =============================================================================
# Derived dimensions (do not change unless you change NODES)
# =============================================================================

INPUT_DIM = 2 * NODES * (NODES - 1)  # = 24 for NODES=4
OUT_DIM   = 2 ** NODES               # = 16 for NODES=4

print(f"NODES={NODES}  TARGET={TARGET}")
print(f"input_dim={INPUT_DIM}  out_dim={OUT_DIM}")
print(f"N_SAMPLES={N_SAMPLES}  NUM_EPOCHS={NUM_EPOCHS}  OPT_STEPS={OPT_STEPS}")

---
## Stage 1 — Data generation

The `generate_dataset()` function:
1. Builds a PyTheus perfect-matching catalogue for an `n`-node graph.
2. Draws random graph weight vectors uniformly from `[-1, 1]`.
3. Computes the **unnormalised** quantum amplitude vector for each weight sample.
4. Saves the data as `.npz` shard files.

**Input shape:** `(N_SAMPLES, INPUT_DIM)` — one weight vector per sample  
**Output shape:** `(N_SAMPLES, OUT_DIM)` — one unnormalised amplitude vector per sample

In [ ]:
from data_generation import generate_dataset

print(f"Generating {N_SAMPLES} samples for {NODES}-node graph...")
t0 = time.perf_counter()

out_dir = generate_dataset(
    vertices     = NODES,
    dimensions   = 2,
    n_samples    = N_SAMPLES,
    batch_size   = DATA_BATCH,
    seed         = DATA_SEED,
    save_to_file = True,
    out_dir_path = str(DATA_DIR),
    shard_size   = SHARD_SIZE,
    normed_data  = NORMED_DATA,
)

elapsed = time.perf_counter() - t0
shard_files = sorted(Path(out_dir).glob("data_*.npz"))

print(f"\nDone in {elapsed:.1f}s")
print(f"Output folder : {out_dir}")
print(f"Shard files   : {len(shard_files)}")
for sf in shard_files:
    d = np.load(sf)
    print(f"  {sf.name}  weights={d['weights'].shape}  amps={d['amps'].shape}")

### Peek at one data sample

Each row of `weights` is a flat vector of graph edge weights.  
Each row of `amps` is the corresponding quantum amplitude vector.

In [ ]:
# Load the first shard and inspect one sample
sample_shard = np.load(shard_files[0])
w0 = sample_shard["weights"][0]
a0 = sample_shard["amps"][0]

print("Sample 0 — graph weights (first 8 of", len(w0), "edges):")
print(" ", np.round(w0[:8], 4))
print()
print("Sample 0 — amplitude vector (all", len(a0), "components):")
print(" ", np.round(a0, 4))
print()
print("Amplitude norm:", round(float(np.linalg.norm(a0)), 6),
      "(≈1.0 only when NORMED_DATA=True; unnormalised otherwise)")

---
## Stage 2 — Shard merging

The training script expects a single `.npz` file.  
`merge_shards_to_npz()` concatenates all shard files into `dataset_merged.npz`.

In [ ]:
from data_generation_utils import merge_shards_to_npz

print("Merging shards...")
t0 = time.perf_counter()

merged_path = merge_shards_to_npz(
    out_dir     = Path(out_dir),
    output_name = "dataset_merged.npz",
)

elapsed = time.perf_counter() - t0

with np.load(merged_path) as d:
    W_shape = d["weights"].shape
    A_shape = d["amps"].shape

print(f"Done in {elapsed:.1f}s")
print(f"Merged file   : {merged_path}")
print(f"weights shape : {W_shape}  (samples × input_dim)")
print(f"amps shape    : {A_shape}  (samples × out_dim)")

MERGED_PATH = str(merged_path)

---
## Stage 3 — Model training

The surrogate model (`PNN` by default) is trained to map graph weights to **unnormalised** amplitude vectors.

**Architecture:**
- `PNN` (Polynomial Neural Network): one hidden layer of size `HIDDEN_DIM` with a monomial activation `x^(n/2)`. Outputs the unnormalised amplitude vector directly.
- `FNN`: a standard multi-layer network (pass a tuple for `HIDDEN_DIM`, e.g. `(400, 400)`).

**Loss:** mean absolute error (MAE) between predicted and true amplitude vectors.

The training config is patched in-notebook so no config file edit is needed.

In [ ]:
import training_config

# Patch the config dict — model_training reads this at module level
training_config.TRAINING_CONFIG.update({
    "NODES"                 : NODES,
    "DIMENSIONS"            : 2,
    "DATE"                  : "sample_workflow",
    "MODEL_NAME"            : MODEL_TYPE,
    "HIDDEN_DIM"            : HIDDEN_DIM,
    "DATA_PATH"             : MERGED_PATH,
    "DATA_SIZE"             : None,
    # NORMALIZE_MODEL_OUTPUT controls normalisation *inside the loss function*.
    # This is independent of NORMED_DATA (which controls the dataset).
    # Here we set it equal to NORMED_DATA for convenience in this demo,
    # but they are separate settings — see WORKFLOW.md for details.
    "NORMALIZE_MODEL_OUTPUT": NORMED_DATA,
    "TRAIN_SPLIT"           : 0.8,
    "VAL_SPLIT"             : 0.1,
    "TEST_SPLIT"            : 0.1,
    "LEARNING_RATE"         : LEARNING_RATE,
    "LR_AFTER_DECAY"        : 1e-5,
    "LR_DECAY_UNTIL_EPOCH"  : NUM_EPOCHS,
    "BATCH_SIZE"            : TRAIN_BATCH,
    "NUM_EPOCHS"            : NUM_EPOCHS,
    "PATIENCE"              : PATIENCE,
    "TOLERANCE"             : 1e-7,
    "INIT_KEY"              : 42,
    "RESUME_FULL_STATE"     : False,
    "CKPT_DIR_RESTORE"      : "",
    "ROOT_FOLDER"           : str(MODEL_DIR),
    "RUN_NAME"              : f"{MODEL_TYPE}_{NODES}_demo",
    "LOSS_NAME"             : "mae",
})

print("Training config set:")
for k in ["NODES", "MODEL_NAME", "HIDDEN_DIM", "NUM_EPOCHS",
           "BATCH_SIZE", "DATA_PATH", "ROOT_FOLDER", "RUN_NAME"]:
    print(f"  {k}: {training_config.TRAINING_CONFIG[k]}")

In [ ]:
import model_training
importlib.reload(model_training)   # picks up the patched config

print(f"Training {MODEL_TYPE} (HIDDEN_DIM={HIDDEN_DIM}, NODES={NODES}) "
      f"for {NUM_EPOCHS} epochs...")
t0 = time.perf_counter()

run_dir = model_training.train_surrogate_model()
run_dir = Path(run_dir)

elapsed = time.perf_counter() - t0
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Run directory: {run_dir}")
print("Output files:")
for f in sorted(run_dir.glob("*")):
    print(f"  {f.name}")

In [ ]:
# Confirm params.msgpack was saved and read back some training info
params_path = run_dir / "params.msgpack"
model_info_path = run_dir / "model_info.json"

assert params_path.exists(), "params.msgpack not found — training may have failed"
print(f"params.msgpack : {params_path.stat().st_size:,} bytes")

with open(model_info_path) as f:
    info = json.load(f)

keys_to_show = ["model_name", "architecture", "input_dims", "output_dims",
                "num_epoch", "batch_size", "best_val_loss", "best_epoch"]
print("\nTraining summary:")
for k in keys_to_show:
    if k in info:
        print(f"  {k}: {info[k]}")

PARAMS_PATH = str(params_path)

---
## Stage 4 — Inverse-design optimisation

The trained surrogate is used as a differentiable proxy for PyTheus.
Gradient descent on the graph weights minimises:

$$\mathcal{L}(x) = (1 - \text{fidelity}(x)) + \lambda_{L1} \|x\|_1$$

The $L_1$ penalty encourages a sparse solution (few active graph edges).
After each optimisation run the graph is **pruned** by zeroing small weights
while keeping fidelity within tolerance.

> **Low fidelity is expected for this tiny demo.**  
> With only 15 training epochs, the surrogate has not converged.  
> The purpose here is to show that the pipeline runs end-to-end.

The full config dict is defined below — no `configs/optimiser_config.py` edit needed.

In [ ]:
from optimiser import run_optimisation

CFG_OPT = {
    # ------------------------------------------------------------------
    # Problem setup
    # ------------------------------------------------------------------
    "n"                     : NODES,
    "dimensions"            : 2,
    "target_name"           : TARGET,
    "model_type"            : MODEL_TYPE,

    # ------------------------------------------------------------------
    # Starting samples
    # generate_data=True → fresh random samples (no file needed)
    # generate_data=False → load from conditioned_data_path
    # ------------------------------------------------------------------
    "generate_data"         : True,
    "conditioned_data_path" : "",
    "max_initial_samples"   : None,
    "data_samples"          : OPT_SAMPLES,
    "data_batch_size"       : OPT_SAMPLES,
    "data_seed"             : 7,
    "low_fidelity_threshold": 0.999,
    "save_generated_data"   : False,
    "data_out_dir"          : str(OPT_DIR / "generated_starts"),
    # normed_data must match NORMED_DATA used in data generation
    # (controls whether starting samples use normalised or unnormalised amplitudes)
    "normed_data"           : NORMED_DATA,
    "generation_gpu_batch_size": OPT_SAMPLES,
    "data_shard_size"       : OPT_SAMPLES,

    # ------------------------------------------------------------------
    # Trained model
    # architecture must match HIDDEN_DIM from training
    # normalize_model_output must match NORMALIZE_MODEL_OUTPUT used in training
    # (independent of normed_data — see WORKFLOW.md normalisation note)
    # ------------------------------------------------------------------
    "architecture"          : HIDDEN_DIM,
    "model_path"            : PARAMS_PATH,
    "normalize_model_output": NORMED_DATA,
    "input_dim"             : INPUT_DIM,
    "out_dim"               : OUT_DIM,

    # ------------------------------------------------------------------
    # Objective and schedule
    # ------------------------------------------------------------------
    "lambda_l1"             : 1e-3,
    "seed"                  : 46,
    "num_steps"             : OPT_STEPS,
    "early_stop_nn_fid"     : 0.9999,
    "learning_rate"         : 1e-2,
    "min_learning_rate"     : 1e-6,
    "lr_decay_steps"        : OPT_STEPS,
    "lr_exponent"           : 1.0,
    "clip_min"              : -1.0,
    "clip_max"              : 1.0,

    # ------------------------------------------------------------------
    # Logging
    # ------------------------------------------------------------------
    "print_every"           : 5,
    "verify_every"          : 5,
    "store_step_vectors"    : False,   # True → large JSON files; keep False here

    # ------------------------------------------------------------------
    # Pruning
    # ------------------------------------------------------------------
    "prune_fid_tolerance"   : 1e-4,
    "prune_thresholds"      : [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],

    # ------------------------------------------------------------------
    # Output
    # ------------------------------------------------------------------
    "results_root"          : str(OPT_DIR),
    "folder_name"           : f"{TARGET}_{NODES}_demo",
    "date"                  : "sample_workflow",
    "k_value"               : 0,
    "noise"                 : 0,
    "CPU_GPU"               : "CPU",
}

print(f"Optimising {OPT_SAMPLES} samples toward {TARGET} (n={NODES}) "
      f"for {OPT_STEPS} steps each...")
t0 = time.perf_counter()

result_dir = run_optimisation(CFG_OPT)
result_dir = Path(result_dir)

elapsed = time.perf_counter() - t0
print(f"\nOptimisation complete in {elapsed:.1f}s")
print(f"Result directory: {result_dir}")

---
## Stage 5 — Load and inspect outputs

Everything the notebook produced lives under `results/sample_workflow/`.  
This section lists the files and prints a short summary.

In [ ]:
# List every file generated by this notebook run
print("=" * 60)
print(f"All files under {RESULTS_ROOT.relative_to(REPO_ROOT)}/")
print("=" * 60)

all_files = sorted(RESULTS_ROOT.rglob("*"))
for p in all_files:
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        rel = p.relative_to(RESULTS_ROOT)
        print(f"  {str(rel):<60s}  {size_kb:6.1f} KB")

In [ ]:
# Read the optimisation summary
summary_file = result_dir / "optimisation_summary.json"
with open(summary_file) as f:
    summary = json.load(f)

print("Optimisation summary")
print("-" * 40)
print(f"Samples run            : {len(summary['final_pytheus_fidelities'])}")
print(f"Steps per sample       : {summary['optimisation_steps']}")
print()
print("PyTheus fidelity to", TARGET, "(higher = better, max = 1.0):")
for i, fid in enumerate(summary["final_pytheus_fidelities"]):
    print(f"  sample {i}: {fid:.4f}")
print()
best_fid = max(summary["final_pytheus_fidelities"])
print(f"Best fidelity in demo  : {best_fid:.4f}")
print()
print("Note: low fidelity is expected for this tiny demo run.")
print("For meaningful results, increase N_SAMPLES, NUM_EPOCHS, and OPT_STEPS.")

In [ ]:
# Print the best-graph solution found
best_file = result_dir / "best_graph_solution.json"
with open(best_file) as f:
    best = json.load(f)

print("Best graph found:")
print(f"  Sample ID      : {best['sample_id']}")
print(f"  Active edges   : {best['num_edges']}  (non-zero after pruning)")
print(f"  PyTheus fidelity: {best['fidelity']:.4f}")
print()
print("Graph weight vector (24 edges for n=4):")
gv = best['graph_vector']
active = [(i, round(v, 4)) for i, v in enumerate(gv) if abs(v) > 1e-4]
print(f"  {len(active)} non-zero weights:")
for idx, val in active:
    print(f"    edge {idx:2d}: {val:+.4f}")

In [ ]:
# Concise pipeline summary
merged_size_kb = Path(MERGED_PATH).stat().st_size / 1024
params_size_kb = Path(PARAMS_PATH).stat().st_size / 1024

print("Pipeline summary")
print("=" * 50)
print(f"Generated dataset : {merged_path.name}  ({merged_size_kb:.0f} KB)")
print(f"  {N_SAMPLES} samples  ·  weights {W_shape}  ·  amps {A_shape}")
print()
print(f"Trained model     : params.msgpack  ({params_size_kb:.0f} KB)")
print(f"  {MODEL_TYPE}  hidden_dim={HIDDEN_DIM}  {NUM_EPOCHS} epochs")
print()
print(f"Optimisation      : {TARGET} (n={NODES})  {OPT_SAMPLES} samples  {OPT_STEPS} steps")
print(f"  Best PyTheus fidelity: {best_fid:.4f}")
print(f"  Results in: {result_dir.relative_to(REPO_ROOT)}")

---
## Cleanup

All files generated by this notebook are inside:

```
results/sample_workflow/
```

To remove them completely:

```bash
rm -rf results/sample_workflow/
```

This folder is already listed in `.gitignore` (`results/`) and will not be committed.

---

## Next steps

Once you have confirmed that the notebook runs end-to-end:

1. **Scale up** — increase `NODES`, `N_SAMPLES`, `NUM_EPOCHS`, and `OPT_STEPS`.
2. **Move to scripts** — for large runs, edit `configs/data_config.py`, `configs/training_config.py`, `configs/optimiser_config.py` and run the pipeline from the terminal.
3. **Use a GPU** — JAX detects and uses a GPU automatically if one is available.
4. **Quick-test script** — `python scripts/run_quick_test.py` verifies the full pipeline in ~10 s without opening any notebook.

See `WORKFLOW.md` for the full data-flow description and production usage notes.